# 03 Walk-Forward Logic Check

Check whether a fixed execution logic remains consistent across out-of-sample windows. This is a bar-level logic check, not a market-realistic execution proof.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = cwd
while not (project_root / "pyproject.toml").exists() and project_root.parent != project_root:
    project_root = project_root.parent
if not (project_root / "pyproject.toml").exists():
    raise RuntimeError("Could not find project root containing pyproject.toml")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

BACKTEST_ROOT = project_root / "backtest_optimize"
RAW_SIGNALS = project_root / "raw_signals"
OUTPUT_DIR = BACKTEST_ROOT / "outputs" / "walkforward_runs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd

from backtest_optimize.contracts import AmbiguityPolicy, MarketSpec
from backtest_optimize.io.signal_loader import load_signal_csv
from backtest_optimize.io.market_data import load_ohlcv_from_core
from backtest_optimize.execution.engine import run_single
from backtest_optimize.analysis.metrics import summarize
from backtest_optimize.analysis.walkforward import (
    make_walkforward_windows,
    run_walkforward,
    split_by_window,
    walkforward_stability_score,
)

pd.set_option("display.max_columns", 100)

In [ ]:
SYMBOL = "US30"
TIMEFRAME = "H4"
SIGNAL_FILE = RAW_SIGNALS / "combo" / "combo_US30_H4_20230102_20260511_signals.csv"
WARMUP_BARS = 0  # Increase when using swing_extreme SL or other lookback-based methods.
MIN_OOS_SIGNALS = 5

MARKET_SPEC = MarketSpec(symbol=SYMBOL, pip_size=1.0, pip_value_per_lot=1.0, min_lot=0.01, lot_step=0.01)

EXECUTION_CONFIG = {
    "account_size": 10_000.0,
    "risk_per_cluster": 0.01,
    "sl_method": "atr_multiple",
    "sl_params": {"atr_mult": 1.5},
    "tp_method": "risk_multiple",
    "tp_params": {"r_multiples": [1.0, 2.0, 3.0]},
    "ambiguity_policy": AmbiguityPolicy.CONSERVATIVE,
    "management": {"sl_move_rule": "breakeven_after_tp1"},
}

In [ ]:
signals = load_signal_csv(SIGNAL_FILE, symbol=SYMBOL, timeframe=TIMEFRAME)
start = signals["bartime"].min()
end = signals["bartime"].max() + pd.Timedelta(days=10)
bars = load_ohlcv_from_core(SYMBOL, TIMEFRAME, start=start, end=end, warmup_bars=WARMUP_BARS, tail_bars=5)

windows = make_walkforward_windows(start=start, end=end, train_months=6, test_months=1, step_months=1)
len(windows)

In [ ]:
def evaluate_window(window):
    _, test_signals = split_by_window(signals, window, time_col="bartime")
    if len(test_signals) < MIN_OOS_SIGNALS:
        return {
            "oos_signal_count": len(test_signals),
            "oos_accepted_count": None,
            "oos_expectancy_r": None,
            "oos_skip_rate": None,
            "oos_ambiguity_rate": None,
            "skip_reason": "below_min_oos_signals",
        }

    result = run_single(
        signals=test_signals,
        bars=bars,
        symbol=SYMBOL,
        timeframe=TIMEFRAME,
        market_spec=MARKET_SPEC,
        **EXECUTION_CONFIG,
    )
    summary = summarize(result)
    return {
        "oos_signal_count": summary["signal_count"],
        "oos_accepted_count": summary["accepted_count"],
        "oos_expectancy_r": summary["expectancy_r"],
        "oos_skip_rate": summary["skip_rate"],
        "oos_ambiguity_rate": summary["ambiguity_rate"],
    }

wf = run_walkforward(evaluate_window, windows)
score = walkforward_stability_score(wf, metric_col="oos_expectancy_r")

display(wf)
print("walkforward_stability_score:", score)

In [ ]:
run_name = f"walkforward_{SYMBOL}_{TIMEFRAME}_{pd.Timestamp.now('UTC').strftime('%Y%m%d_%H%M%S')}"
path = OUTPUT_DIR / f"{run_name}.csv"
wf.to_csv(path, index=False)
path